In [105]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import countDistinct
filepath="D:\Khushan_BDA\ABD\SparkSession\Data\sf-fire-calls.csv"

In [106]:
def create_sparkSession():
    spark=SparkSession.builder.appName('Fire Example').getOrCreate()
    return spark

In [107]:
def create_dataframe(spark,filepath):
    df=spark.read.csv(filepath,header=True,inferSchema=True)
    df1=df.select('CallType','CallDate','City','Zipcode','Neighborhood','Delay')
    return df1

In [108]:
def clean_dataset(df):
    df1=df.withColumn('Date',to_date(col('CallDate'), 'MM/dd/yyyy')).drop('CallDate')
    df2=df1.withColumn('Year',year(col('Date')))\
    .withColumn('Month',month(col('Date')))\
    .withColumn('week',weekofyear(col('Date')))
    return df2

In [109]:
spark=create_sparkSession()
df=create_dataframe(spark,filepath)
df=clean_dataset(df)
df.printSchema()

root
 |-- CallType: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Delay: double (nullable = true)
 |-- Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- week: integer (nullable = true)



In [110]:
df.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|week|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94102|          Tenderloin|1.7833333|2002-01-11|2002|    1|   2|
|Medical I

In [111]:
#create a user defined function
def mapSeason(data):
    if  2 < data < 6:
        return 'Spring'
    elif 5 < data < 9:
        return 'Summer'
    elif 8 < data <12:
        return 'Autumn'
    else:
        return 'Winter'
seasonUDF=udf(mapSeason,StringType())
clean_df=df.withColumn('Season',seasonUDF(col('Month')))
clean_df.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|week|Season|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|Winter|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|Winter|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94102

In [112]:
# 1. Get yearly count of fire calls
clean_df.select('Year').groupBy('Year').count().orderBy('year',ascending=True).show()

+----+-----+
|Year|count|
+----+-----+
|2000| 5459|
|2001| 7713|
|2002| 8090|
|2003| 8499|
|2004| 8283|
|2005| 8282|
|2006| 8174|
|2007| 8255|
|2008| 8869|
|2009| 8789|
|2010| 9341|
|2011| 9735|
|2012| 9674|
|2013|10020|
|2014|10775|
|2015|11458|
|2016|11609|
|2017|12135|
|2018|10136|
+----+-----+



In [113]:
# 2. What were all the different types of fire calls in 2018?
clean_df.select('CallType').where(col('Year')==2018).distinct().show(truncate=False)

+-------------------------------+
|CallType                       |
+-------------------------------+
|Elevator / Escalator Rescue    |
|Alarms                         |
|Odor (Strange / Unknown)       |
|Citizen Assist / Service Call  |
|HazMat                         |
|Vehicle Fire                   |
|Other                          |
|Outside Fire                   |
|Traffic Collision              |
|Assist Police                  |
|Gas Leak (Natural and LP Gases)|
|Water Rescue                   |
|Electrical Hazard              |
|Structure Fire                 |
|Medical Incident               |
|Fuel Spill                     |
|Smoke Investigation (Outside)  |
|Train / Rail Incident          |
|Explosion                      |
|Suspicious Package             |
+-------------------------------+



In [114]:
# 3. Which week in the year in 2018 had the most fire calls?
clean_df.select('Week')\
    .where(col('Year')==2018)\
    .groupBy('Week')\
    .count()\
    .orderBy('count',ascending=False).collect()[0][0]

22

In [115]:
clean_df.where(col('Year') == 2018) \
    .groupBy('Week') \
    .count() \
    .collect()

[Row(Week=44, count=244),
 Row(Week=6, count=225),
 Row(Week=3, count=224),
 Row(Week=5, count=236),
 Row(Week=9, count=228),
 Row(Week=4, count=202),
 Row(Week=8, count=232),
 Row(Week=39, count=224),
 Row(Week=7, count=228),
 Row(Week=10, count=232),
 Row(Week=45, count=64),
 Row(Week=38, count=226),
 Row(Week=11, count=240),
 Row(Week=36, count=203),
 Row(Week=12, count=221),
 Row(Week=13, count=243),
 Row(Week=16, count=228),
 Row(Week=40, count=255),
 Row(Week=20, count=225),
 Row(Week=19, count=233),
 Row(Week=41, count=220),
 Row(Week=15, count=222),
 Row(Week=37, count=223),
 Row(Week=17, count=203),
 Row(Week=21, count=231),
 Row(Week=14, count=225),
 Row(Week=18, count=236),
 Row(Week=31, count=234),
 Row(Week=34, count=232),
 Row(Week=28, count=231),
 Row(Week=26, count=223),
 Row(Week=27, count=223),
 Row(Week=22, count=259),
 Row(Week=1, count=246),
 Row(Week=43, count=250),
 Row(Week=35, count=221),
 Row(Week=23, count=235),
 Row(Week=25, count=249),
 Row(Week=24, count=1

In [116]:
# Max Month
#max_month.select('Week','count').filter(col('count')==max_month.agg({'count':'max'}).collect()[0][0]).collect()[0][0]

In [117]:
# 4. Get monthly count of fire calls based on year

monthly_count = clean_df.groupBy('Year', 'Month') \
    .count() \
    .orderBy('Year', 'Month')

monthly_count.show()

+----+-----+-----+
|Year|Month|count|
+----+-----+-----+
|2000|    4|  335|
|2000|    5|  680|
|2000|    6|  585|
|2000|    7|  668|
|2000|    8|  678|
|2000|    9|  655|
|2000|   10|  620|
|2000|   11|  595|
|2000|   12|  643|
|2001|    1|  622|
|2001|    2|  613|
|2001|    3|  692|
|2001|    4|  636|
|2001|    5|  682|
|2001|    6|  672|
|2001|    7|  646|
|2001|    8|  660|
|2001|    9|  577|
|2001|   10|  673|
|2001|   11|  619|
+----+-----+-----+
only showing top 20 rows


In [118]:
# 5. Monthly report of fire call types for selected year

selected_year = 2018

monthly_calltype_report = clean_df \
    .filter(col('Year') == selected_year) \
    .groupBy('Month', 'CallType') \
    .count() \
    .orderBy('Month', desc('count'))

monthly_calltype_report.show(truncate=False)

+-----+-------------------------------+-----+
|Month|CallType                       |count|
+-----+-------------------------------+-----+
|1    |Medical Incident               |692  |
|1    |Alarms                         |122  |
|1    |Structure Fire                 |91   |
|1    |Traffic Collision              |42   |
|1    |Citizen Assist / Service Call  |15   |
|1    |Outside Fire                   |14   |
|1    |Gas Leak (Natural and LP Gases)|5    |
|1    |Water Rescue                   |4    |
|1    |Vehicle Fire                   |4    |
|1    |Other                          |3    |
|1    |Electrical Hazard              |3    |
|1    |Smoke Investigation (Outside)  |3    |
|1    |Elevator / Escalator Rescue    |3    |
|1    |Train / Rail Incident          |2    |
|1    |Odor (Strange / Unknown)       |2    |
|1    |Fuel Spill                     |1    |
|1    |HazMat                         |1    |
|2    |Medical Incident               |635  |
|2    |Alarms                     

In [119]:
# 6. Top five fire call types for every season of selected year

from pyspark.sql.window import Window

selected_year = 2018

season_calltypes = clean_df \
    .filter(col('Year') == selected_year) \
    .groupBy('Season', 'CallType') \
    .count()

windowSpec = Window.partitionBy('Season') \
    .orderBy(desc('count'))

top_five_season = season_calltypes \
    .withColumn('Rank', row_number().over(windowSpec)) \
    .filter(col('Rank') <= 5) \
    .orderBy('Season', 'Rank')

top_five_season.show(truncate=False)

+------+-----------------+-----+----+
|Season|CallType         |count|Rank|
+------+-----------------+-----+----+
|Autumn|Medical Incident |1514 |1   |
|Autumn|Alarms           |251  |2   |
|Autumn|Structure Fire   |201  |3   |
|Autumn|Traffic Collision|100  |4   |
|Autumn|Outside Fire     |39   |5   |
|Spring|Medical Incident |2110 |1   |
|Spring|Alarms           |333  |2   |
|Spring|Structure Fire   |261  |3   |
|Spring|Traffic Collision|133  |4   |
|Spring|Other            |36   |5   |
|Summer|Medical Incident |2053 |1   |
|Summer|Alarms           |336  |2   |
|Summer|Structure Fire   |262  |3   |
|Summer|Traffic Collision|121  |4   |
|Summer|Outside Fire     |61   |5   |
|Winter|Medical Incident |1327 |1   |
|Winter|Alarms           |224  |2   |
|Winter|Structure Fire   |182  |3   |
|Winter|Traffic Collision|79   |4   |
|Winter|Outside Fire     |28   |5   |
+------+-----------------+-----+----+



In [120]:
# 7. Check whether fire type calls are seasonal

seasonal_calls = clean_df \
    .groupBy('Season', 'CallType') \
    .count() \
    .orderBy('Season', desc('count'))

seasonal_calls.show(truncate=False)

+------+-------------------------------+-----+
|Season|CallType                       |count|
+------+-------------------------------+-----+
|Autumn|Medical Incident               |28569|
|Autumn|Structure Fire                 |5921 |
|Autumn|Alarms                         |4980 |
|Autumn|Traffic Collision              |1858 |
|Autumn|Citizen Assist / Service Call  |646  |
|Autumn|Other                          |565  |
|Autumn|Outside Fire                   |461  |
|Autumn|Vehicle Fire                   |208  |
|Autumn|Gas Leak (Natural and LP Gases)|192  |
|Autumn|Water Rescue                   |173  |
|Autumn|Odor (Strange / Unknown)       |133  |
|Autumn|Smoke Investigation (Outside)  |122  |
|Autumn|Electrical Hazard              |122  |
|Autumn|Elevator / Escalator Rescue    |105  |
|Autumn|HazMat                         |49   |
|Autumn|Fuel Spill                     |48   |
|Autumn|Industrial Accidents           |21   |
|Autumn|Explosion                      |20   |
|Autumn|Water

In [121]:
# 8. Months in 2018 with the highest number of fire calls

monthly_2018 = clean_df \
    .filter(col('Year') == 2018) \
    .groupBy('Month') \
    .count() \
    .orderBy(desc('count'))

monthly_2018.show()

+-----+-----+
|Month|count|
+-----+-----+
|   10| 1068|
|    5| 1047|
|    3| 1029|
|    8| 1021|
|    1| 1007|
|    6|  974|
|    7|  974|
|    9|  951|
|    4|  947|
|    2|  919|
|   11|  199|
+-----+-----+



In [122]:
# 9. Find major calltype in each year

year_calltype_count = clean_df \
    .groupBy('Year', 'CallType') \
    .count()

windowSpec = Window.partitionBy('Year') \
    .orderBy(desc('count'))

major_calltype = year_calltype_count \
    .withColumn('Rank', row_number().over(windowSpec)) \
    .filter(col('Rank') == 1) \
    .orderBy('Year')

major_calltype.show(truncate=False)

+----+----------------+-----+----+
|Year|CallType        |count|Rank|
+----+----------------+-----+----+
|2000|Medical Incident|3408 |1   |
|2001|Medical Incident|4653 |1   |
|2002|Medical Incident|5046 |1   |
|2003|Medical Incident|5056 |1   |
|2004|Medical Incident|5137 |1   |
|2005|Medical Incident|5084 |1   |
|2006|Medical Incident|5027 |1   |
|2007|Medical Incident|5114 |1   |
|2008|Medical Incident|5692 |1   |
|2009|Medical Incident|5671 |1   |
|2010|Medical Incident|6186 |1   |
|2011|Medical Incident|6413 |1   |
|2012|Medical Incident|6296 |1   |
|2013|Medical Incident|6690 |1   |
|2014|Medical Incident|7176 |1   |
|2015|Medical Incident|7812 |1   |
|2016|Medical Incident|7999 |1   |
|2017|Medical Incident|8330 |1   |
|2018|Medical Incident|7004 |1   |
+----+----------------+-----+----+



In [123]:
# 10. Average delay in response for each call type

avg_delay = clean_df \
    .groupBy('CallType') \
    .agg(
        avg('Delay').alias('Average_Delay')
    ) \
    .orderBy(desc('Average_Delay'))

avg_delay.show(truncate=False)

+-----------------------------------+------------------+
|CallType                           |Average_Delay     |
+-----------------------------------+------------------+
|Mutual Aid / Assist Outside Agency |38.416666311111115|
|Assist Police                      |26.981903994285716|
|Train / Rail Incident              |16.452046763157895|
|Administrative                     |12.261111333333332|
|HazMat                             |7.527016126612904 |
|Marine Fire                        |6.928571314285715 |
|Confined Space / Structure Collapse|6.915384576923078 |
|Watercraft in Distress             |6.886904817857142 |
|Suspicious Package                 |6.57666672        |
|High Angle Rescue                  |6.048958375000001 |
|Water Rescue                       |5.507748342145695 |
|Other                              |5.505155432421977 |
|Fuel Spill                         |5.49222798238342  |
|Citizen Assist / Service Call      |5.473342576604596 |
|Electrical Hazard             

In [124]:
# 11. Find calltype with maximum average delay

max_avg_delay = clean_df \
    .groupBy('CallType') \
    .agg(
        avg('Delay').alias('Average_Delay')
    ) \
    .orderBy(desc('Average_Delay'))

max_avg_delay.limit(1).show(truncate=False)

+----------------------------------+------------------+
|CallType                          |Average_Delay     |
+----------------------------------+------------------+
|Mutual Aid / Assist Outside Agency|38.416666311111115|
+----------------------------------+------------------+



In [125]:
# 12. Neighborhood with most fire calls in 2018

neighborhood_2018 = clean_df \
    .filter(col('Year') == 2018) \
    .groupBy('Neighborhood') \
    .count() \
    .orderBy(desc('count'))

neighborhood_2018.show(truncate=False)

+------------------------------+-----+
|Neighborhood                  |count|
+------------------------------+-----+
|Tenderloin                    |1393 |
|South of Market               |1053 |
|Mission                       |913  |
|Financial District/South Beach|772  |
|Bayview Hunters Point         |522  |
|Western Addition              |352  |
|Sunset/Parkside               |346  |
|Nob Hill                      |295  |
|Hayes Valley                  |291  |
|Outer Richmond                |262  |
|Castro/Upper Market           |251  |
|North Beach                   |231  |
|Excelsior                     |212  |
|West of Twin Peaks            |210  |
|Potrero Hill                  |210  |
|Chinatown                     |191  |
|Pacific Heights               |191  |
|Marina                        |191  |
|Mission Bay                   |178  |
|Bernal Heights                |170  |
+------------------------------+-----+
only showing top 20 rows


In [126]:
# 13. Neighborhoods with worst response times in 2018

worst_response = clean_df \
    .filter(
        (col('Year') == 2018) &
        col('Neighborhood').isNotNull()
    ) \
    .groupBy('Neighborhood') \
    .agg(
        avg('Delay').alias('Average_Delay')
    ) \
    .orderBy(desc('Average_Delay'))

worst_response.show(truncate=False)

+------------------------------+------------------+
|Neighborhood                  |Average_Delay     |
+------------------------------+------------------+
|Chinatown                     |6.190314097905761 |
|Presidio                      |5.8292270414492755|
|Treasure Island               |5.4537037125      |
|McLaren Park                  |4.744047642857142 |
|Bayview Hunters Point         |4.620561956877393 |
|Presidio Heights              |4.594131472394366 |
|Inner Sunset                  |4.438095199935065 |
|Inner Richmond                |4.364728682713179 |
|Financial District/South Beach|4.3440846182901565|
|Haight Ashbury                |4.266428599285713 |
|Seacliff                      |4.261111146666666 |
|West of Twin Peaks            |4.190952390857142 |
|Potrero Hill                  |4.190555557428571 |
|Pacific Heights               |4.180453718900524 |
|Tenderloin                    |4.101519516597274 |
|Oceanview/Merced/Ingleside    |3.947242180719424 |
|Excelsior  

In [127]:
# 14. Analyze trend of average response delay for each call type

yearly_delay = clean_df \
    .groupBy('Year', 'CallType') \
    .agg(
        avg('Delay').alias('Average_Delay')
    ) \
    .orderBy('CallType', 'Year')

yearly_delay.show(truncate=False)

+----+------------------+------------------+
|Year|CallType          |Average_Delay     |
+----+------------------+------------------+
|2005|Administrative    |31.983334         |
|2006|Administrative    |1.8               |
|2017|Administrative    |3.0               |
|2000|Aircraft Emergency|3.905555533333333 |
|2001|Aircraft Emergency|2.616666675       |
|2002|Aircraft Emergency|4.14666662        |
|2003|Aircraft Emergency|13.166667         |
|2004|Aircraft Emergency|2.5916667         |
|2005|Aircraft Emergency|4.29166675        |
|2006|Aircraft Emergency|3.2111111166666664|
|2007|Aircraft Emergency|3.094444333333333 |
|2009|Aircraft Emergency|3.0083335         |
|2011|Aircraft Emergency|3.5944443333333336|
|2012|Aircraft Emergency|4.65833335        |
|2013|Aircraft Emergency|2.4333334         |
|2014|Aircraft Emergency|7.75              |
|2015|Aircraft Emergency|1.1333333         |
|2000|Alarms            |3.0111468393086813|
|2001|Alarms            |2.6232156389407737|
|2002|Alar

In [128]:
# 15. Find which city has more calltypes for each year

city_calltypes = clean_df \
    .groupBy('Year', 'City') \
    .agg(
        countDistinct('CallType').alias('CallType_Count')
    )

windowSpec = Window.partitionBy('Year') \
    .orderBy(desc('CallType_Count'))

top_city_each_year = city_calltypes \
    .withColumn('Rank', row_number().over(windowSpec)) \
    .filter(col('Rank') == 1) \
    .orderBy('Year')

top_city_each_year.show(truncate=False)

+----+-------------+--------------+----+
|Year|City         |CallType_Count|Rank|
+----+-------------+--------------+----+
|2000|SF           |18            |1   |
|2001|SF           |20            |1   |
|2002|SF           |20            |1   |
|2003|SF           |24            |1   |
|2004|SF           |23            |1   |
|2005|SF           |26            |1   |
|2006|SF           |24            |1   |
|2007|SF           |26            |1   |
|2008|SF           |23            |1   |
|2009|SF           |22            |1   |
|2010|SF           |23            |1   |
|2011|SF           |25            |1   |
|2012|SF           |25            |1   |
|2013|SF           |24            |1   |
|2014|San Francisco|21            |1   |
|2015|San Francisco|24            |1   |
|2016|San Francisco|24            |1   |
|2017|San Francisco|26            |1   |
|2018|San Francisco|20            |1   |
+----+-------------+--------------+----+



In [129]:
# 16. Find top 5 cities based on number of fire calls

top_5_cities = clean_df \
    .groupBy('City') \
    .count() \
    .orderBy(desc('count')) \
    .limit(5)

top_5_cities.show()

+-------------+------+
|         City| count|
+-------------+------+
|           SF|120072|
|San Francisco| 51739|
|SAN FRANCISCO|  1676|
|           TI|   486|
|     Presidio|   281|
+-------------+------+



In [130]:
# 17. Correlation between Zipcode and number of fire calls

zipcode_calls = clean_df \
    .filter(col('Zipcode').isNotNull()) \
    .groupBy('Zipcode') \
    .count() \
    .withColumnRenamed('count', 'Fire_Call_Count')

zipcode_calls.show()

+-------+---------------+
|Zipcode|Fire_Call_Count|
+-------+---------------+
|  94109|          14686|
|  94115|           7812|
|  94112|           8421|
|  94127|           1881|
|  94108|           4084|
|  94121|           4555|
|  94105|           4236|
|  94131|           3236|
|  94116|           3933|
|  94134|           5009|
|  94124|           9236|
|  94102|          21840|
|  94114|           5175|
|  94107|           6941|
|  94111|           2974|
|  94103|          20897|
|  94117|           5804|
|  94122|           6355|
|  94110|          14801|
|  94132|           4321|
+-------+---------------+
only showing top 20 rows
